# Loan Default Prediction Model
End-to-end pipeline: load → clean → feature engineer → train → evaluate → save

## 1. Imports & Config

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, joblib, time
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    RocCurveDisplay, PrecisionRecallDisplay, average_precision_score
)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import lightgbm as lgb

pd.set_option('display.max_columns', 50)
SEED = 42
DATA_PATH = 'loan.csv'
print('Libraries loaded OK')

Libraries loaded OK


## 2. Load Data

In [2]:
t0 = time.time()
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Loaded in {time.time()-t0:.1f}s  |  shape: {df_raw.shape}')
df_raw['loan_status'].value_counts()

Loaded in 179.0s  |  shape: (2260668, 145)


loan_status
Fully Paid                                             1041952
Current                                                 919695
Charged Off                                             261655
Late (31-120 days)                                       21897
In Grace Period                                           8952
Late (16-30 days)                                         3737
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     31
Name: count, dtype: int64

## 3. Define Target — Binary Default Flag

In [ ]:
# Keep only closed loans — exclude 'Current' since outcome is unknown
KEEP_STATUSES = {
    'Fully Paid': 0,
    'Charged Off': 1,
    'Default': 1,
    'Late (31-120 days)': 1,
    'Late (16-30 days)': 1,
    'Does not meet the credit policy. Status:Charged Off': 1,
    'Does not meet the credit policy. Status:Fully Paid': 0,
    'In Grace Period': 1,  # treat as at-risk
}

df = df_raw[df_raw['loan_status'].isin(KEEP_STATUSES)].copy()
df['default'] = df['loan_status'].map(KEEP_STATUSES)

print(f'Rows after filtering: {len(df):,}')
print(f"Default rate: {df['default'].mean():.2%}")
df['default'].value_counts()

## 4. Feature Selection — Blacklist Leaky Columns

In [ ]:
# Drop post-loan columns (not available at origination) and admin/id columns
# This is safer than a whitelist — new useful columns are automatically included

# Exact post-loan payment columns
LEAKAGE_COLS = [
    'out_prncp', 'out_prncp_inv',
    'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
    'recoveries', 'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d',
    'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low',
    'funded_amnt_inv',
    # Admin / identifier columns — no predictive value
    'id', 'member_id', 'url', 'desc', 'title',
    # Target column — keep separately
    'loan_status',
]

# Pattern-based drop — catches entire families of leaky columns
LEAKAGE_PATTERNS = [
    'hardship_',      # e.g. hardship_flag, hardship_amount, hardship_start_date
    'settlement_',    # e.g. settlement_status, settlement_date, settlement_amount
    'total_rec_',     # e.g. total_rec_prncp, total_rec_int
    'debt_settlement_',
]

pattern_drops = [
    c for c in df.columns
    if any(c.startswith(p) for p in LEAKAGE_PATTERNS)
]

all_drops = list(set(LEAKAGE_COLS + pattern_drops))
all_drops = [c for c in all_drops if c in df.columns]

df2 = df.drop(columns=all_drops)

print(f'Dropped {len(all_drops)} leaky/admin columns')
print(f'Remaining columns: {df2.shape[1]}  (includes target \'default\')')
print(f'\nDropped columns:')
for c in sorted(all_drops): print(f'  - {c}')


## 5. Feature Engineering

In [ ]:
df2 = df2.copy()  # already filtered by blacklist above

# -- term: '36 months' → 36
df2['term'] = df2['term'].str.extract(r'(\d+)').astype(float)

# -- int_rate: '15.02%' → 15.02
if df2['int_rate'].dtype == object:
    df2['int_rate'] = df2['int_rate'].str.replace('%','').astype(float)

# -- revol_util: '73.2%' → 73.2
if df2['revol_util'].dtype == object:
    df2['revol_util'] = df2['revol_util'].str.replace('%','').astype(float)

# -- emp_length: '10+ years' → 10, '< 1 year' → 0
def parse_emp(s):
    if pd.isna(s): return np.nan
    if '10+' in str(s): return 10
    if '< 1' in str(s): return 0
    digits = ''.join(filter(str.isdigit, str(s)))
    return int(digits) if digits else np.nan

df2['emp_length'] = df2['emp_length'].apply(parse_emp)

# -- earliest_cr_line → credit age in months from 2020
df2['cr_line_age_mths'] = pd.to_datetime(df2['earliest_cr_line'], errors='coerce')
df2['cr_line_age_mths'] = ((pd.Timestamp('2020-01-01') - df2['cr_line_age_mths'])
                            .dt.days / 30).round(1)
df2.drop(columns=['earliest_cr_line'], inplace=True)

# -- Derived ratios
df2['loan_to_income']     = df2['loan_amnt'] / (df2['annual_inc'] + 1)
df2['installment_to_inc'] = df2['installment'] / (df2['annual_inc'] / 12 + 1)

# -- Encode ordinals
GRADE_MAP = {'A':7,'B':6,'C':5,'D':4,'E':3,'F':2,'G':1}
df2['grade_num'] = df2['grade'].map(GRADE_MAP)

# -- Low-cardinality categoricals → LabelEncoder
CAT_COLS = ['home_ownership','verification_status','purpose',
            'initial_list_status','sub_grade','addr_state']
CAT_COLS = [c for c in CAT_COLS if c in df2.columns]

le = LabelEncoder()
for col in CAT_COLS:
    df2[col] = df2[col].fillna('MISSING')
    df2[col] = le.fit_transform(df2[col].astype(str))

# Drop original grade (replaced by grade_num)
df2.drop(columns=['grade'], inplace=True, errors='ignore')

print('Feature engineering done')
print(f'Final shape: {df2.shape}')
df2.head(3)

## 6. Missing Value Overview

In [ ]:
miss = df2.isnull().mean().sort_values(ascending=False)
miss = miss[miss > 0]
print(f'{len(miss)} columns with missing values:')
print(miss.head(20).to_string())

# Drop columns with >50% missing
high_miss = miss[miss > 0.5].index.tolist()
print(f'\nDropping {len(high_miss)} columns with >50% missing: {high_miss}')
df2.drop(columns=high_miss, inplace=True)

## 7. Train / Validation / Test Split

In [ ]:
X = df2.drop(columns=['default'])
y = df2['default']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=SEED, stratify=y_train
)

print(f'Train: {X_train.shape[0]:,}  Val: {X_val.shape[0]:,}  Test: {X_test.shape[0]:,}')
print(f'Default rate — Train: {y_train.mean():.2%}  Val: {y_val.mean():.2%}  Test: {y_test.mean():.2%}')

## 8. Train LightGBM Model

In [ ]:
params = {
    'objective': 'binary',
    'metric': 'auc',
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'min_child_samples': 50,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'class_weight': 'balanced',
    'random_state': SEED,
    'n_jobs': -1,
    'verbose': -1,
}

model = lgb.LGBMClassifier(**params)

t0 = time.time()
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)]
)
print(f'\nTraining time: {time.time()-t0:.1f}s')
print(f'Best iteration: {model.best_iteration_}')

## 9. Evaluate on Test Set

In [ ]:
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = model.predict(X_test)

roc_auc = roc_auc_score(y_test, y_prob)
avg_prec = average_precision_score(y_test, y_prob)

print(f'ROC-AUC:           {roc_auc:.4f}')
print(f'Avg Precision:     {avg_prec:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['No Default','Default']))

## 10. ROC & Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], name='LightGBM')
axes[0].set_title(f'ROC Curve (AUC = {roc_auc:.3f})')
axes[0].plot([0,1],[0,1],'k--', alpha=0.5)

PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1], name='LightGBM')
axes[1].set_title(f'Precision-Recall (AP = {avg_prec:.3f})')

plt.tight_layout()
plt.savefig('roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Default','Default'],
            yticklabels=['No Default','Default'], ax=ax)
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
ax.set_title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Feature Importance

In [ ]:
importances = pd.Series(model.feature_importances_, index=X_train.columns)
top30 = importances.nlargest(30).sort_values()

fig, ax = plt.subplots(figsize=(8, 10))
top30.plot.barh(ax=ax, color='steelblue')
ax.set_title('Top 30 Feature Importances (LightGBM)')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10:')
print(importances.nlargest(10).to_string())

## 13. Default Risk Score Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y_prob[y_test == 0], bins=60, alpha=0.6, label='No Default', color='steelblue')
ax.hist(y_prob[y_test == 1], bins=60, alpha=0.6, label='Default',    color='tomato')
ax.set_xlabel('Predicted Default Probability')
ax.set_ylabel('Count')
ax.set_title('Risk Score Distribution')
ax.legend()
plt.tight_layout()
plt.savefig('score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Threshold Analysis — Precision vs Recall Tradeoff

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)

best_idx = f1_scores.argmax()
best_threshold = thresholds[best_idx]
print(f'Best F1 threshold: {best_threshold:.3f}')
print(f'  Precision: {precision[best_idx]:.3f}')
print(f'  Recall:    {recall[best_idx]:.3f}')
print(f'  F1:        {f1_scores[best_idx]:.3f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(thresholds, precision[:-1], label='Precision', color='steelblue')
ax.plot(thresholds, recall[:-1],    label='Recall',    color='tomato')
ax.plot(thresholds, f1_scores[:-1], label='F1',        color='green', linestyle='--')
ax.axvline(best_threshold, color='black', linestyle=':', label=f'Best threshold={best_threshold:.2f}')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs Threshold')
ax.legend()
plt.tight_layout()
plt.savefig('threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Predict on New Loans (Inference Example)

In [ ]:
# Simulate inference on 5 test examples
sample = X_test.head(5).copy()
sample['actual_default']  = y_test.head(5).values
sample['default_prob']    = model.predict_proba(X_test.head(5))[:, 1].round(3)
sample['predicted_label'] = (sample['default_prob'] >= best_threshold).astype(int)
sample['risk_tier'] = pd.cut(
    sample['default_prob'],
    bins=[0, 0.2, 0.4, 0.6, 1.0],
    labels=['Low', 'Medium', 'High', 'Very High']
)

display_cols = ['default_prob', 'predicted_label', 'actual_default', 'risk_tier']
sample[display_cols]

## 16. Save Model & Artifacts

In [ ]:
# Save the model
joblib.dump(model, 'loan_default_model.pkl')

# Save feature list so inference uses same columns
joblib.dump(list(X_train.columns), 'model_features.pkl')

# Save threshold
joblib.dump(best_threshold, 'best_threshold.pkl')

print('Saved:')
print('  loan_default_model.pkl')
print('  model_features.pkl')
print('  best_threshold.pkl')

# --- Quick reload test ---
loaded = joblib.load('loan_default_model.pkl')
feats  = joblib.load('model_features.pkl')
thresh = joblib.load('best_threshold.pkl')
prob   = loaded.predict_proba(X_test[feats].head(1))[:, 1][0]
print(f'\nReload test — prob on first test row: {prob:.3f}  (threshold: {thresh:.3f})')
print('Model ready for deployment.')